# Reddit Logs to YTMusic Playlists

## Setup

In [1]:
YEAR=2025

# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..\\browser.json'

# Path to .tsv reddit search logs filtered to just new entries
reddit_log_path = '..\\..\\reddit-scraper\\db'
search_db_path = '..\\..\\reddit-scraper\\logs'

manual_labels_file = 'reddit_all_manual_labels.tsv'

## Imports / Helpers

In [2]:
import os
import glob
import time
import csv

import time
import random


import unicodedata
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

from datetime import date


### DataFrame Helpers

In [3]:

match_tsv_col_order = ['manual_label', 'ytmusic_key', 'reddit_title', 'match_quality',
       'match_score_token_set_ratio', 'match_score_token_sort_ratio',
       'is_album', 'reddit_sub', 'ytmusic_album', 'ytmusic_albumId',
       'ytmusic_artist', 'ytmusic_artistId', 'ytmusic_title',
       'ytmusic_videoId', 'ytmusic_duration', 'ytmusic_year',
       'ytmusic_resultType', 'reddit_key', 'reddit_post_id', 'reddit_sub_id',
       'reddit_aggregator', 'reddit_source_url', 'youtube_videoId']


def save_df_to_tsv(df, base_file_name, search_db_path):
    """
    Save the given DataFrame to a TSV file in the specified path with a name including the current date.
    """
    today_str = date.today().strftime('%Y-%m-%d')
    file_name = f'{base_file_name}_{today_str}.tsv'
    full_path = os.path.join(search_db_path, 'ytmusic', file_name)
    df.to_csv(full_path, sep='\t', header=True)
    print(f'Successfully saved DataFrame with shape: {df.shape} to {full_path}')


def load_tsv_to_df(file_name, search_db_path):
    """
    Load a TSV file into a DataFrame.
    FIX: Added low_memory=False to prevent DtypeWarning on mixed-type columns like the index.
    """
    db_tsv_path = os.path.join(search_db_path, 'ytmusic', file_name)
    db = pd.read_csv(db_tsv_path, sep='\t', index_col=0, low_memory=False)
    if 'reddit_sub' in db.columns and 'reddit_source_url' in db.columns and 'reddit_post_id' not in db.columns:
        db['reddit_post_id'] = db.reddit_sub + '//' + db.reddit_source_url
    print(f'Loaded {len(db)} entries from {db_tsv_path}')
    return db


def remove_duplicates(df, key='reddit_post_id', keep='first', check_inconsitent_manual_label=False):
    """
    Remove duplicate rows based on the specified key and check for 'manual_label' inconsistencies.
    """
    if key not in df.columns:
        print(f"Warning: Key '{key}' not found in DataFrame. No duplicates removed.")
        return df
    
    if check_inconsitent_manual_label:
        # Check for inconsistencies in 'manual_label' for duplicates
        duplicates = df[df.duplicated(key, keep=keep)]
        inconsistent = duplicates.groupby(key).filter(lambda x: x['manual_label'].nunique() > 1)
        if not inconsistent.empty:
            print(f"Warning: Inconsistent 'manual_label' values found for some keys: {inconsistent[key].unique()}")

    # Remove duplicates and print the number of duplicates being removed
    before_removal = len(df)
    df = df.drop_duplicates(subset=key)
    after_removal = len(df)
    print(f'Removed {before_removal - after_removal} duplicate entries based on {key}.')

    return df


### YTMusic API and Functions

In [4]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(lambda x: x[0]['id'])
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(ytmusic_header_path)


yt_res_cache = {}
yt_unmatched_cache = {}

### String Helper Functions

In [5]:
def scrub_title(title):
    orig_title = title
    title = str(title).lower().strip()
    title = unicodedata.normalize('NFKD', title).encode('ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    title = title.replace('| ', '(')
    # remove stuff at end of title
    for k in [
         'official video', 'music video', 'live video','lyric video', 'cover)', 'video)', 'prod.', 'produced by', 
         'album stream', 'album review', 'album version', 'full album','produced by', 'npr music tiny desk concert',
         'anniversary expanded edition']:
        if k in title:
            new_t = title.split(k)[0]
            if len(new_t) > 5:
                title = title.split(k)[0]
    start_char = ['[', '('] 
    for k in [
        'official', 'unoffical', 'free', 'explicit', 'video', 'music', 'nsfw', 'original', 'lyric', 'studio', 'vinyl',
        'full', 'album)', 'audio)', 'cover)', 'convert', 'thissongissick', 'duploc', 'prod', 'leak', 'from', 'lofi hip', 
        'remaster', 'uncensored', '720p', '1080p', '320k' 'repackag', '19', '20', 'dir', 'quality upgrade', 'complete', 
        'with lyrics', 'visualizer', 'deluxe', 'anniversary edition']:
        for s in start_char:
            t = s + k
            if t in title:
                new_t = title.split(t)[0]
                if len(new_t) > 5:
                    title = title.split(t)[0]
    for s in start_char:
        if title.endswith(s):
            title = title[:-1]     
    # remove tokens from title
    for k in ['[hd]', '[hq]', 'hd', 'hq', '()', '[]', ' | ', '{}']:
        if k in title:
            title=title.replace(k, '')
    if title.endswith(' - '):
        title = title[0:-3]
    return title
    

def check_album_scrub_title(title):
    # check for album
    title = title.lower().strip()
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True

    title = scrub_title(title)
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## Parse Reddit .tsv and query YTMusic for Match

* Now checks db tsv to see if ialready matched (basd on sub and url)
* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 12/14/2025




In [6]:
db = load_tsv_to_df(manual_labels_file, search_db_path)
prev_match_ids = frozenset(db['reddit_post_id'])
# print(f'{prev_match_ids} previous entries in set')

Loaded 107018 entries from ..\..\reddit-scraper\logs\ytmusic\reddit_all_manual_labels.tsv


In [ ]:
reddit_subfolders = [f'new_{YEAR}']
log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 2


log_tsvs = []
for folder in reddit_subfolders:
    tsv_path = os.path.join(reddit_log_path, folder)
    log_tsvs += list(glob.glob(os.path.join(tsv_path, '*.tsv')))
log_tsvs = sorted(log_tsvs)
print(f'Found {len(log_tsvs)} reddit tsvs')


matched_entries = []
unmatched_entries = []
t0 = time.time()
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']
for i, tsv_file in enumerate(log_tsvs, start=1):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg = toks

    # df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    try:
        df = pd.read_csv(tsv_file, sep='\t', index_col=0, engine='python', quotechar='"')
    except Exception as e:
        print(f"CRITICAL: Failed to parse {tsv_file}. Error: {e}")
        continue # Skip this broken file
      
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    # assert set(df.columns) == set(expected_cols)
    # --- REPLACEMENT FOR THE ASSERT ---
    actual_cols = set(df.columns)
    expected_set = set(expected_cols)

    if actual_cols != expected_set:
        print(f'\n\n!!! CRITICAL: Column mismatch in file: {name}.tsv !!!')
        
        missing_cols = expected_set - actual_cols
        extra_cols = actual_cols - expected_set

        print(f"  - Actual columns found ({len(actual_cols)}): {list(df.columns)}")
        print(f"  - Expected columns ({len(expected_set)}): {expected_cols}")

        if missing_cols:
            print(f"  - >>> Columns MISSING from the file: {list(missing_cols)}")
        
        if extra_cols:
            print(f"  - >>> EXTRA columns found in the file: {list(extra_cols)}")
            
        print("  Skipping this file due to schema mismatch.\n")
        continue
    # --- END OF REPLACEMENT ---
    
    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = check_album_scrub_title(entry.title)
        # print(f"DEBUG: Preparing to search for: '{title_key}' from original title: '{entry.title}'")

        # If already in db get match from there
        post_id = f'{sub}//{entry.url}'
        if post_id in prev_match_ids:
            continue
        # Check cache for saved YTMusic query response or previous match failures        
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_album', ''))}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_title', ''))}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_post_id'] = post_id
                unmatch['reddit_sub_id'] = f'{sub}//{entry.url}'
                unmatch['reddit_aggregator'] = agg
                unmatch['manual_label'] = 'no-match'
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            # match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_title_len'] = len(str(entry.title))
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_post_id'] = post_id
        match['reddit_sub_id'] = f'{sub}//{entry.url}'
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id
        match['manual_label'] = f'no-label_{date.today()}'

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Using manual grading to set thresholds
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 60:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 75:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')


        # Update match results
        matched_entries.append(match)
print(f'\nFinished matching in {time.time() - t0 // 60:0.1f} minutes')
# Process Matched entries
match_df = pd.DataFrame(matched_entries)[match_tsv_col_order]
match_df = remove_duplicates(match_df, key='reddit_sub_id', keep='first')
save_df_to_tsv(match_df,  f'reddit_{YEAR}-new_ytmusic_scored_new_matches.tsv', search_db_path)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
unmatch_df = remove_duplicates(unmatch_df, key='reddit_sub_id', keep='first')
save_df_to_tsv(unmatch_df,  f'reddit_{YEAR}-new_ytmusic_failed_new_matches.tsv', search_db_path)

## Manually grade


### 2024 version (archive)

In [ ]:
# # 02024 version
# # https://docs.google.com/spreadsheets/d/1Z6X6rmOoLTqo3na_8Owrdf2O6FJ08n71buPhiP5uawA/edit#gid=2087986722

# new_f = 'reddit_2024-new_ytmusic_scored_new_matches.tsv_2024-12-31.tsv'
# new_matches =  load_tsv_to_df(new_f, search_db_path)

# match_f = 'reddit_all_manual_labels.tsv'
# graded_matches =  load_tsv_to_df(match_f, search_db_path)

# print(f"graded_matches shape {graded_matches.shape}")
# print(f"new_matches shape {new_matches.shape}")

# new_matches_filtered = new_matches[
#     ~new_matches["reddit_title"].isin(graded_matches["reddit_title"])
# ]

# print(
#     f"new_matches_filtered shape after removing duplicates: {new_matches_filtered.shape}"
# )

# print(new_matches_filtered['match_quality'].value_counts())


# graded_tsv_file = 'reddit_2024-new_ytmusic_scored_new_matches__model-graded_2024-1-5.tsv'


# # TODO try this graded_matches = load_tsv_to_df(graded_tsv_file, search_db_path)
# graded_matches =  load_tsv_to_df(graded_tsv_file, search_db_path)
# # graded_matches = pd.read_csv(os.path.join(search_db_path, 'ytmusic', graded_tsv_file), sep='\t')
# graded_matches = graded_matches.sort_values('reddit_sub')


# # all_manual_labels = load_tsv_to_df(manual_labels_file, search_db_path)
# # manual_labels = pd.concat([
# #   graded_matches[all_manual_labels.columns], 
# #   all_manual_labels
# # ]).sort_values(['manual_label','ytmusic_key'], ascending=False)

# # Replace old manual labels
# # print(f'Loaded history with shape: {manual_labels.shape}')
# # manual_labels = remove_duplicates(manual_labels, check_inconsitent_manual_label=True)
# # print(f'Loaded and de-duped history, now has shape: {manual_labels.shape} and manual labels:\n{manual_labels.manual_label.value_counts()}')
# # save_df_to_tsv(manual_labels,  f'{manual_labels_file}_new_.tsv', search_db_path)
# # print(f'Manually overwrite _new over {manual_labels_file}')


### 2025

used gemini to grade, `see notes.md`

In [8]:
# --- Main Logic to Update the Historical Master File ---

# --- 1. Define File Names and Paths ---
graded_2025_filename = 'reddit_2025-new_ytmusic_scored_new_matches__gemini-graded_2025-12-13.tsv'
historical_master_filename = 'reddit_all_manual_labels.tsv'
historical_master_path = os.path.join(search_db_path, 'ytmusic', historical_master_filename)

# --- 2. Load, Standardize, and Merge Data ---
df_2025 = load_tsv_to_df(graded_2025_filename, search_db_path)
df_historical_before = load_tsv_to_df(historical_master_filename, search_db_path) # Keep an original copy for the "before" plot

print("\nStandardizing labels...")
label_map = {
    'pass': 'passed-match', 'passed-match': 'passed-match',
    'fail': 'failed-match', 'failed-match': 'failed-match',
    'no-match': 'failed-match'
}
df_2025['manual_label'] = df_2025['manual_label'].map(label_map)
df_historical = df_historical_before.copy() # Work with a copy to preserve the original
df_historical['manual_label'] = df_historical['manual_label'].map(label_map)

print("Merging data...")
common_cols = [col for col in df_historical.columns if col in df_2025.columns]
df_merged = pd.concat([df_historical, df_2025[common_cols]])

# --- 3. De-duplicate, Sort, and Finalize DataFrame ---
print("De-duplicating by keeping the highest-scoring match...")
duplicate_subset = ['ytmusic_key', 'reddit_key', 'reddit_sub']
df_merged.sort_values('match_score_token_set_ratio', ascending=False, inplace=True)
df_final_master = df_merged.drop_duplicates(subset=duplicate_subset, keep='first')
print(f"Removed {len(df_merged) - len(df_final_master)} duplicate entries.")

print("Sorting and reordering columns...")
df_final_master = df_final_master.sort_values(
    by=['reddit_sub', 'manual_label', 'match_score_token_set_ratio'],
    ascending=[True, True, False]
)
df_final_master = df_final_master[match_tsv_col_order]

# --- 4. Backup Original and Save New Master File ---
print("\nBacking up and saving the final master file...")
backup_year = '2025'
backup_path = f"{os.path.splitext(historical_master_path)[0]}_before_{backup_year}.tsv"
if not os.path.exists(backup_path):
    os.rename(historical_master_path, backup_path)
    print(f"  - Backed up original file to: {os.path.basename(backup_path)}")
else:
    print(f"  - Backup file already exists, skipping rename.")


df_final_master.to_csv(historical_master_path, sep='\t', index=True)
print(f"  - Successfully overwrote master file: {os.path.basename(historical_master_path)}")

# --- 5. Create 'passing' and 'failing' Objects for Subsequent Cells ---
print("Standardizing 'reddit_sub' column to lowercase...")
# df_final_master['reddit_sub'] = df_final_master['reddit_sub'].str.lower()

print("\nCreating 'passing' and 'failing' objects for next cell...")
passing = df_final_master[df_final_master['manual_label'] == 'passed-match'].copy()
failing = df_final_master[df_final_master['manual_label'] == 'failed-match'].copy()
print(f"Final data objects created: {len(df_final_master)} total, {len(passing)} passing, {len(failing)} failing.")



Loaded 9411 entries from ..\..\reddit-scraper\logs\ytmusic\reddit_2025-new_ytmusic_scored_new_matches__gemini-graded_2025-12-13.tsv
Loaded 107018 entries from ..\..\reddit-scraper\logs\ytmusic\reddit_all_manual_labels.tsv

Standardizing labels...
Merging data...
De-duplicating by keeping the highest-scoring match...
Removed 9411 duplicate entries.
Sorting and reordering columns...

Backing up and saving the final master file...
  - Backup file already exists, skipping rename.
  - Successfully overwrote master file: reddit_all_manual_labels.tsv
Standardizing 'reddit_sub' column to lowercase...

Creating 'passing' and 'failing' objects for next cell...
Final data objects created: 107018 total, 69879 passing, 37139 failing.


## Make Playlists

In [9]:
import os
import sys
path_backup = os.path.join('../')
module_path = os.path.abspath(path_backup)
if module_path not in sys.path:
    sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists
import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')


RUN_API_AUTH_TEST = True
HEADER_FILE = path_backup + 'browser.json'
PLAYLIST_TSV_DIR = path_backup + 'playlists/'

PLAYCOUNT_FILE='_ytmusic_lastfm_playcount.tsv'
NOT_LIKE_PLAYLIST_TSV ='_not_liked_tracks.tsv'


Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

passing_albums = passing.loc[passing['is_album'] == True]
# passing_tracks = passing.loc[passing.is_album == False]
passing_tracks = passing.loc[passing['is_album'] == False] # may be str may be bool

# Create a temporary, normalized (lowercase) column in the dataframe.
# We will group by this to combine variations like '50sMusic' and '50smusic'.
passing_tracks['normalized_sub'] = passing_tracks['reddit_sub'].str.lower()
passing_albums['normalized_sub'] = passing_albums['reddit_sub'].str.lower()

# Create a pandas Series of existing YTMusic playlist titles, also in lowercase.
# This will be our fast, case-insensitive lookup table.
yt_playlist_titles_lower = Y.playlists['title'].str.lower()


Using ytmusicapi version: 1.11.3
Using header file: ../browser.json
-> Method: Browser Authentication (cookies). Skipping OAuth client init.
Test Passed in 4.03 seconds
Using ytmusicapi version: 1.11.3
Loaded 509 playlists


C:\Users\jake\AppData\Local\Temp\ipykernel_15820\2082623983.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passing_tracks['normalized_sub'] = passing_tracks['reddit_sub'].str.lower()
C:\Users\jake\AppData\Local\Temp\ipykernel_15820\2082623983.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passing_albums['normalized_sub'] = passing_albums['reddit_sub'].str.lower()


## Subreddit playlists for passing tracks

(same code for round 1 and 2, just clear completed [])

will update if playlist exists, otherwise create a new radio for subreddit

In [10]:

# Use this if it gets stuck to get completed back to where it was
completed = []
# stop_sub = 'idm'
# n_subs = len(passing_tracks['reddit_sub'].unique())
# for sub, df in passing_tracks.groupby('reddit_sub'):

#   completed.append(sub)
#   if sub == stop_sub:
#     print(f'Stopped at {stop_sub}')
#     break

# print(f'So far completed {len(completed)} of {n_subs} ({len(completed)/n_subs:0.1%}%)')


In [ ]:
# when you get banned it seem like it is for around 25 playlists, ban last 6-8 hrs?
# SLEEP_TIME=6*60*60
# print(f'Sleeping {SLEEP_TIME//60//60}hrs')
# time.sleep(SLEEP_TIME)
# 

# completed = []
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not


from pandas import merge


SLEEP_TIME = 5
MIN_N_LIKE = 5
DRY = False



# --- 2. Refactored Main Loop ---

# Group by the normalized subreddit name to process all case variations together.
track_sub_grp = passing_tracks.groupby('normalized_sub')
n_subs = len(track_sub_grp)
for i, (normalized_sub, df_group) in enumerate(track_sub_grp,start=1):

    # Get all original case variations for this group (e.g., {'50sMusic', '50smusic'})
    original_subs_in_group = set(df_group['reddit_sub'].unique())
    
    # Check if ANY of the original subreddit names have already been completed.
    if not original_subs_in_group.isdisjoint(completed):
        print(f"Group '{normalized_sub}' contains an already completed sub. Skipping.")
        continue

    # --- Find the correct YTMusic playlist (Case-Insensitive) ---
    target_title_lower = f'xr {normalized_sub} radio'
    playlist_exists = target_title_lower in yt_playlist_titles_lower.values
    
    actual_title = None
    pl_id = -1

    if playlist_exists:
        # Find the original, correctly-cased title from YTMusic
        match_index = yt_playlist_titles_lower[yt_playlist_titles_lower == target_title_lower].index[0]
        actual_title = Y.playlists['title'].loc[match_index]
        pl_id = Y.playlists['playlistId'].loc[match_index]
    else:
        # For new playlists, we'll use the FIRST original subreddit name as the convention.
        actual_title = f'xr {df_group["reddit_sub"].iloc[0]} radio'

    # --- Prepare Video IDs ---
    vids_raw = df_group.ytmusic_videoId.dropna().unique().tolist()
    vids = list(frozenset(vids_raw) - Y.banned_vid_set)
    
    if not vids:
        print(f"No valid, non-banned video IDs found for group '{normalized_sub}'. Skipping.")
        continue
    
    print(f'\n({i}/{n_subs}) Processing {len(vids)} tracks for playlist: {actual_title}')

    # --- Create or Update Playlist ---
    if playlist_exists:
        if not DRY:
            ytm.add_playlist_items(playlistId=pl_id, videoIds=vids, duplicates=False)
        print(f'Updated {len(vids)} tracks in existing playlist with id: {pl_id}')
    else:
        desc = f'Matched {len(vids)} tracks for r/{normalized_sub} using filters: {df_group.reddit_aggregator.unique()}'
        if not DRY:
            pl_id = ytm.create_playlist(title=actual_title, description=desc, privacy_status='PRIVATE', video_ids=vids)
        print(f'Created new playlist with {len(vids)} tracks and id: {pl_id}')
    
    # --- Post-Action Steps ---
    sleep_time = SLEEP_TIME + random.randint(1, 6)
    print(f'Waiting {sleep_time} seconds...')
    if not DRY:
        time.sleep(sleep_time)

    if not DRY and pl_id != -1:
        Y.clean_up_radio_playlist(
            pl_info=Y.playlist_get_info(pl_id, use_cache=True),
            verbose=True, sleep=1,
            move_like=MIN_N_LIKE, create_like_playlist=True,
            remove_dislike=True, remove_not_like=True
        )
   
      # too much on API
      # Y.playcount_sort_playlist(Y.playlist_get_info(pl_id, use_cache=False), ignore_banned=True)

    # Add ALL original subreddit names from this group to the completed list
    completed.extend(original_subs_in_group)
    print(f"Group '{normalized_sub}' processing complete. Updated 'completed' list.")
    


Group '60smusic' contains an already completed sub. Skipping.
Group '90salternative' contains an already completed sub. Skipping.
Group '90shiphop' contains an already completed sub. Skipping.
Group '90srock' contains an already completed sub. Skipping.
Group 'afropop' contains an already completed sub. Skipping.
Group 'aggrotech' contains an already completed sub. Skipping.
Group 'ambientmusic' contains an already completed sub. Skipping.
Group 'americana' contains an already completed sub. Skipping.
Group 'artpop' contains an already completed sub. Skipping.
Group 'bedroompop' contains an already completed sub. Skipping.
Group 'bigbeat' contains an already completed sub. Skipping.
Group 'blackmetal' contains an already completed sub. Skipping.
Group 'boybands' contains an already completed sub. Skipping.
Group 'breakbeat' contains an already completed sub. Skipping.
Group 'britpop' contains an already completed sub. Skipping.
Group 'burial' contains an already completed sub. Skipping

### Playlist Merge

anual + Gemini effort to map

TODO add these to _mapping playlist

In [13]:
# --- Cell to get and print all playlists with track counts ---

# Make sure your Y object from ytmusic_library is initialized first
# Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)

print("Fetching and printing all playlists with track counts...")

# The Y.playlists DataFrame should already be loaded during initialization.
# It contains 'title' and 'count' columns.
# Select the relevant columns
playlists_df = Y.playlists[['title', 'count']].copy()
# Sort alphabetically by playlist title, case-insensitive
playlists_df = playlists_df.sort_values(by='title', key=lambda col: col.str.lower())
# playlists_df = playlists_df.sort_values(by='count')

# # Print the DataFrame without the index for a clean list
print(playlists_df.to_string(index=False))


Fetching and printing all playlists with track counts...
                                    title count
                               2022 Recap   NaN
                               2023 Recap   NaN
                               2024 Recap   NaN
                          ambiant electro    65
                                  ambient   899
                              ambient BOC    41
             ambient Dream Pop Deep Sleep    47
                      ambient drone metal    22
                          ambient grouper    72
              ambient haunting harmonious   139
                     ambient Indie Synths    84
                            ambient japan    50
                           ambient modern    49
                     ambient modern radio    45
                   ambient modular synths    36
                            ambient radio   275
                              Archive Mix   NaN
                                    beats 1,296
               beats 2010s wonk

In [11]:
# Format: "Source Playlist Exact Name": "Target Playlist Exact Name"
merge_map = {
  #  to xr indierock radio
#  to 
    # ==========================================
    # 1. HIP HOP, RAP & BEATS
    # ==========================================
    "xr hiphopheads": "xr hiphop",
    "xr hiphopheads radio": "xr hiphop radio",
    "xr rap radio": "xr hiphop radio",
    "xr grime radio":"hiphop modern radio",
    "xr Gfunk": "Hip Hop 1990s west Coast",
    "xr Gfunk radio": "Hip Hop 1990s west Coast radio", 
    "xr 90shiphop radio": "Hip Hop 1990s radio",
    "xr futurebeats": "future beats",
    "xr futurebeats radio": "future beats radio",
    "xr lofihiphop radio": "hiphop modern radio",
    "xr trap radio": "xr trapmuzik radio",
    "xr hiphopheadsnorthwest radio": "xr hiphop radio",

    # ==========================================
    # 2. ROCK, PUNK & METAL
    # ==========================================
    # --- Garage & Rock ---
    "xr garagerock radio": "rock garage radio",
    "xr garagepunk": "rock garage",
    "xr garagepunk radio": "rock garage radio",
    "xr ClassicRock": "rock classic",
    "xr ClassicRock radio": "rock classic radio",
    "xr alternativerock radio": "xr Rock radio",
    "xr 90sRock radio": "rock 1990s radio",
    "xr 90sAlternative": "rock 1990s alternative",
    "xr mathrock radio": "xr indierock radio",
    "xr noiserock radio": "rock garage radio",
    
    # --- Punk & New Wave ---
    "xr postpunk": "Post-Punk 1970s-1980s",
    "xr postpunk radio": "Post-Punk 1970s-1980s radio", 
    "xr NewWave": "Rock 1980s Pop New Wave",
    "xr NewWave radio": "Rock 1980s New Wave radio",
    "xr coldwave radio": "Rock 1980s New Wave radio", 
    "xr darkwave radio": "Rock 1980s New Wave radio",
    "xr skapunk radio": "xr PunkRock radio",
    "xr Punk-Rock radio": "xr PunkRock radio",

    # --- The Metal Collapse ---
    "xr industrialmetal radio": "xr metal radio",
    "xr symphonicmetal radio": "xr metal radio",
    "xr grindcore radio": "xr metal radio",
    "xr screamo radio": "xr metal radio",
    "xr progmetal radio": "xr metal radio",
    "xr powermetal radio": "xr metal radio",
    "xr deathcore radio": "xr metal radio",
    "xr posthardcore radio": "xr metal radio",
    "xr djent radio": "xr metal radio",
    "xr metalcore radio": "xr metal radio",
    "xr thrashmetal radio": "xr metal radio",
    "xr vikingmetal radio": "xr metal radio",
    "xr folkmetal radio": "xr metal radio",
    "xr deathmetal radio": "xr metal radio",
    "xr speedmetal radio": "xr metal radio",
    "xr blackmetal radio": "xr metal radio",
    "xr postmetal radio": "xr metal radio",
    "xr heavymetal radio": "xr metal radio",
    
    # ==========================================
    # 3. ELECTRONIC & DANCE
    # ==========================================
    "xr breakbeat radio": "electronic Garage Bass House radio",
    "xr futuregarage": "future garage",
    "xr futurebass": "future bass",
    "xr brostep radio": "xr dubstep radio", 
    "xr tech-house radio": "xr house radio",
    "xr burial radio": "xr futuregarage radio",
    "xr codingmusic radio": "electronic Focus radio",
    "xr runningmusic": "electronic Focus",
    "xr runningmusic radio": "electronic Focus radio",
    "xr aggrotech radio": "xr theOverload radio",
    "xr hyperpop radio": "xr futurebass radio",
    "xr dnb radio": "xr AtmosphericDnB radio",
    "xr jungle radio": "xr AtmosphericDnB radio",
    "xr spacebass radio": "xr bassheavy radio",
    "xr bigbeat radio": "electronic big beats radio",
    "xr witch-house radio": "xr witchHouse radio",
    "xr OldskoolRave": "xr OldElectronicMusic",
    "xr stonerrock": "rock stoner sludge dank",
    "xr stonerrock radio": "rock stoner sludge dank radio",


    # ==========================================
    # 4. INDIE, FOLK & POP
    # ==========================================
    "xr indieheads radio": "xr indie radio",
    "xr bedroompop radio": "xr indie albums",
    "xr shoegaze": "Shoegaze",
    "xr shoegaze radio": "Shoegaze radio",
    "xr slowcore radio": "post rock slow core radio",
    "xr americana radio":  "Folk radio",
    "xr folk radio": "Folk radio",
    "xr boybands radio": "xr 2000smusic radio",
    "xr britpop radio": "xr popheads radio",
    "xr artpop radio": "xr popheads radio",

    # ==========================================
    # 5. SOUL, JAZZ & VIBES
    # ==========================================
    "xr FunkSouMusic": "soul funk",
    "xr FunkSouMusic radio": "soul radio",
    "xr neosoul radio": "soul radio",
    "xr motown radio": "soul motown radio",
    "xr modernjazz radio": "xr jazz radio",
    "xr nujazz radio": "xr jazz radio",
    "xr darkjazz radio": "xr jazznoir radio",
    "xr freejazz radio": "xr jazz radio",

    # ==========================================
    # 6. AMBIENT & CHILL
    # ==========================================
    "xr ambientmusic": "ambient", 
    "xr ambientmusic radio": "ambient radio",
    "xr rainymood": "ambient modern", 
    "xr rainymood radio": "ambient modern radio",
    "xr lostwave radio": "xr VintageObscura radio",
    "xr vaporwave radio": "xr chillwave radio",

    # ==========================================
    # 7. WORLD / FOREIGN
    # ==========================================
    "xr afropop radio": "xr AfricanMusic radio",
    "xr cpop radio": "xr kpop radio",
    "xr latinpopheads radio": "xr WorldMusic radio",
    "xr japanesemusic radio": "xr WorldMusic radio",
    "xr italianmusic radio": "xr WorldMusic radio",
    "xr indianindie radio": "xr WorldMusic radio",
    "xr frenchrap radio": "xr foreignrap radio",
    "xr germanrap radio": "xr foreignrap radio",
    "xr ska radio": "Reggae radio",
    "xr dancehall radio": "Reggae radio",
    "xr reggae radio": "Reggae radio",

    # ==========================================
    # 8. DECADES / MISC
    # ==========================================
    "xr newmusic radio": "xr 2020smusic radio",
    "xr 60sMusic radio": "oldies 1960s radio",
}

In [60]:
import time
import random
import pandas as pd

# --- Configuration ---
SLEEP_TIME_BETWEEN_MERGES = 4
DRY_RUN = False 

# Ensure 'completed' list exists (run completed = [] in a previous cell to reset)
if 'completed' not in locals():
    completed = []

# --- Execution ---
start_time = time.time()
stats = {"merged": 0, "tracks_added": 0, "deleted": 0, "skipped": 0, "errors": 0}

print(f"Starting SMART-DEDUPE Consolidation with Resume Support...")
print(f"Total Rules: {len(merge_map)} | Already Completed: {len(completed)}")

# Refresh Library
Y.playlists = pd.DataFrame(Y.yt.get_library_playlists(limit=5000))
title_to_id = dict(zip(Y.playlists['title'].str.lower(), Y.playlists['playlistId']))

for i, (source_title, target_title) in enumerate(merge_map.items(), 1):
    
    # 0. Check Resume List
    if source_title in completed:
        # print(f"[{i}] Skipping '{source_title}' (Already in completed list).")
        stats["skipped"] += 1
        continue

    src_lower = source_title.lower()
    tgt_lower = target_title.lower()

    # 1. Validation
    if src_lower not in title_to_id:
        print(f"[{i}] Source '{source_title}' not found in library. Marking as complete.")
        stats["skipped"] += 1
        completed.append(source_title) # It's gone, so we are done with it
        continue
    
    if tgt_lower not in title_to_id:
        print(f"[{i}] WARN: Target '{target_title}' not found. Skipping.")
        stats["errors"] += 1
        continue

    src_id = title_to_id[src_lower]
    tgt_id = title_to_id[tgt_lower]
    
    if src_id == tgt_id:
        completed.append(source_title)
        continue

    print(f"\n[{i}/{len(merge_map)}] Processing: '{source_title}' -> '{target_title}'")

    try:
        # 2. Fetch Source Tracks
        src_data = Y.playlist_get_info(src_id, use_cache=False)
        src_tracks = src_data.get('tracks', []) if isinstance(src_data, dict) else src_data
        src_vids = []
        if isinstance(src_tracks, list):
            src_vids = [t['videoId'] for t in src_tracks if 'videoId' in t]
        elif isinstance(src_tracks, pd.DataFrame) and 'videoId' in src_tracks.columns:
            src_vids = src_tracks['videoId'].dropna().unique().tolist()

        if not src_vids:
            print(f"   Source is empty. Deleting...")
            if not DRY_RUN:
                Y.yt.delete_playlist(src_id)
                stats["deleted"] += 1
                completed.append(source_title) # Success
            continue

        # 3. Fetch Target Tracks (For Client-Side Deduplication)
        tgt_data = Y.playlist_get_info(tgt_id, use_cache=False)
        tgt_tracks = tgt_data.get('tracks', []) if isinstance(tgt_data, dict) else tgt_data
        tgt_vids_set = set()
        if isinstance(tgt_tracks, list):
            tgt_vids_set = set(t['videoId'] for t in tgt_tracks if 'videoId' in t)
        elif isinstance(tgt_tracks, pd.DataFrame) and 'videoId' in tgt_tracks.columns:
            tgt_vids_set = set(tgt_tracks['videoId'].dropna().unique())

        # 4. Calculate Difference
        # Only add tracks that are NOT in the target
        vids_to_add = [v for v in src_vids if v not in tgt_vids_set]
        
        duplicates_count = len(src_vids) - len(vids_to_add)
        
        if duplicates_count > 0:
            print(f"   Skipping {duplicates_count} duplicates found locally.")

        success = False

        if not vids_to_add:
            print(f"   All {len(src_vids)} tracks already exist in target.")
            # If all are duplicates, we can safely delete source
            if not DRY_RUN:
                print(f"   Deleting source '{source_title}'...")
                Y.yt.delete_playlist(src_id)
                stats["deleted"] += 1
                stats["merged"] += 1
                success = True
            else:
                success = True # Simulate success for dry run
        else:
            print(f"   Adding {len(vids_to_add)} NEW tracks to target...")
            if not DRY_RUN:
                # We do NOT pass duplicates=False here because we already deduped locally.
                status = Y.yt.add_playlist_items(playlistId=tgt_id, videoIds=vids_to_add)
                
                # Check for success
                if status and isinstance(status, dict) and status.get('status') == 'STATUS_SUCCEEDED':
                    stats["tracks_added"] += len(vids_to_add)
                    print(f"   Success. Deleting source '{source_title}'...")
                    Y.yt.delete_playlist(src_id)
                    stats["deleted"] += 1
                    stats["merged"] += 1
                    success = True
                else:
                    print(f"   !!! FAIL: API Error: {status}")
                    stats["errors"] += 1
            else:
                print("   [DRY RUN] Would add unique tracks and delete source.")
                success = True

        # 5. Update Completed List
        if success:
            completed.append(source_title)

    except Exception as e:
        print(f"   CRITICAL ERROR processing '{source_title}': {e}")
        stats["errors"] += 1
        continue

    if not DRY_RUN:
        time.sleep(SLEEP_TIME_BETWEEN_MERGES + random.randint(1, 3))

# --- Report ---
print("\n" + "="*50)
print(f"Processed Rules: {len(merge_map)} | Newly Merged: {stats['merged']}")
print(f"Tracks Added: {stats['tracks_added']} | Errors: {stats['errors']}")
print(f"Total Completed Playlists: {len(completed)}")
print("="*50)

Starting SMART-DEDUPE Consolidation with Resume Support...
Total Rules: 97 | Already Completed: 0
[1] Source 'xr hiphopheads' not found in library. Marking as complete.

[2/97] Processing: 'xr hiphopheads radio' -> 'xr hiphop radio'
   Skipping 98 duplicates found locally.
   Adding 8 NEW tracks to target...
   Success. Deleting source 'xr hiphopheads radio'...

[3/97] Processing: 'xr rap radio' -> 'xr hiphop radio'
   Skipping 2 duplicates found locally.
   Adding 3 NEW tracks to target...
   Success. Deleting source 'xr rap radio'...

[4/97] Processing: 'xr grime radio' -> 'hiphop modern radio'
   Adding 2 NEW tracks to target...
   Success. Deleting source 'xr grime radio'...

[5/97] Processing: 'xr Gfunk' -> 'Hip Hop 1990s west Coast'
   Skipping 21 duplicates found locally.
   All 21 tracks already exist in target.
   Deleting source 'xr Gfunk'...
[6] WARN: Target 'Hip Hop 1990s west Coast radio' not found. Skipping.

[7/97] Processing: 'xr 90shiphop radio' -> 'Hip Hop 1990s radio

#### Analysis of the Run

**Status:** **Excellent / High Success Rate.**
*   **Successes:** You successfully consolidated **84 playlists** and moved **3,453 tracks**. This is a massive reduction in clutter.
*   **The "Fail" Cases:** The 3 errors you saw (`xr NewWave radio`, `xr futurebeats radio`, `xr 60sMusic radio`) were all due to the same issue: **"One or more of the tracks are already in your playlist"**.
    *   **Why?** Even though we did client-side deduplication (Python checking `if vid in target`), the YouTube API sometimes flags "duplicates" based on track metadata (Title/Artist) even if the Video IDs are technically different (e.g., Album version vs Single version).
    *   **Result:** The API threw a `STATUS_FAILED` asking for a confirmation dialog.
    *   **Manual Fix:** Since you handled these manually, the result is perfect. The script did its job of protecting you by *not* deleting the source when the API complained.

---

In [66]:
import time
import random
import pandas as pd

# --- Configuration ---
MIN_N_LIKE = 5 # Move to 'Like' playlist if > 5 likes found
MIN_TRACK_COUNT = 30 # Only process if playlist is larger than this
SLEEP_TIME = 5 # Seconds between heavy operations

# --- Setup Targets ---
# We only want to process the UNIQUE targets from the map
target_playlists = list(set(merge_map.values()))
target_playlists.sort()

# History tracking to resume if needed
if 'processed_targets' not in locals():
    processed_targets = []

print(f"Starting Final Cleanup & Sort on {len(target_playlists)} Target Playlists...")

# Refresh Library Cache one last time to get new counts/IDs
print("Refreshing Library...")
Y.playlists = pd.DataFrame(Y.yt.get_library_playlists(limit=5000))
# Map Title -> ID
lib_lookup = dict(zip(Y.playlists['title'].str.lower(), Y.playlists['playlistId']))

for i, pl_title in enumerate(target_playlists, 1):

    if pl_title in processed_targets:
        continue

    print(f"\n[{i}/{len(target_playlists)}] Checking: '{pl_title}'")

    # [NEW] 0. Skip non-"radio" playlists
    if not pl_title.lower().endswith(' radio'):
        print(f"   Skipping: Title does not end in 'radio'.")
        processed_targets.append(pl_title)
        continue
    
    # 1. Get Playlist ID
    pl_title_lower = pl_title.lower()
    if pl_title_lower not in lib_lookup:
        print(f"   Skipping: Playlist not found in library (maybe renamed?).")
        processed_targets.append(pl_title)
        continue
        
    pl_id = lib_lookup[pl_title_lower]

    try:
        # 2. Get Info (No Cache) to check count
        pl_info = Y.playlist_get_info(pl_id, use_cache=False)
        track_count = pl_info.get('trackCount', len(pl_info.get('tracks', [])))
        
        if track_count < MIN_TRACK_COUNT:
            print(f"   Skipping: Count {track_count} is below threshold ({MIN_TRACK_COUNT}).")
            processed_targets.append(pl_title)
            continue

        print(f"   Processing {track_count} tracks...")

        # 3. Clean Up (Remove Dislikes / Move Likes)
        # Note: This returns a counter dict
        cleanup_stats = Y.clean_up_radio_playlist(
            pl_info=pl_info,
            verbose=True, 
            sleep=1,
            move_like=True, # Enable moving likes
            min_num_like=MIN_N_LIKE,
            create_like_playlist=True,
            remove_dislike=True, 
            remove_not_like=True
        )
        
        # If tracks were removed, the local pl_info is slightly stale, 
        # but sort_playlist fetches fresh data anyway.

        # 4. Sort by Playcount
        # Note: This usually DELETEs and RE-CREATES the playlist
        print(f"   Sorting by Playcount...")
        Y.playcount_sort_playlist(
            Y.playlist_get_info(pl_id, use_cache=False), 
            ignore_banned=True
        )
        
        # Mark done
        processed_targets.append(pl_title)
        
        # Sleep to avoid rate limits
        time.sleep(SLEEP_TIME + random.randint(2, 5))

    except Exception as e:
        print(f"   CRITICAL ERROR on '{pl_title}': {e}")
        # We do NOT mark as processed so you can retry
        time.sleep(10)

print("\n" + "="*50)
print(f"Cleanup Job Complete.")
print(f"Processed: {len(processed_targets)} / {len(target_playlists)}")
print("="*50)

Starting Final Cleanup & Sort on 62 Target Playlists...
Refreshing Library...

[22/62] Checking: 'future beats radio'
   Processing 1567 tracks...
Radio playlist future beats radio counters: {'moved_like': 26, 'removed_not_like': 7}
Moved 26 LIKE entries from future beats radio to PLWptjpDqazOyX9N3j5lVFIPy68A2JrLTV
Removed 7 NOT_LIKE entries from future beats radio
   Sorting by Playcount...
Created sorted pl: future beats radio [12-19-2025] PLWptjpDqazOyLQNffYGQqgFemOOqiAVsv, and  deleted original pl: PLWptjpDqazOxaaLR1KNq0MACtDVKtQ08R

[32/62] Checking: 'rock garage radio'
   Processing 413 tracks...
Radio playlist rock garage radio counters: {'moved_like': 13}
Moved 13 LIKE entries from rock garage radio to PLWptjpDqazOzpaPg4RVIlY-jia-yBQBdI
   Sorting by Playcount...
Created sorted pl: rock garage radio [12-19-2025] PLWptjpDqazOybslH_nByH3Ty3jzgryk-2, and  deleted original pl: PLWptjpDqazOxPkJjVhusJUHDTC4P4pfcI

[33/62] Checking: 'soul funk'
   Skipping: Title does not end in 'radi

## Subreddit playlists for passing albums 
(same code for round 1 and 2, just clear completed [])

In [12]:
completed= []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']
# stop_sub = 'idm'
# n_subs = len(passing_tracks['reddit_sub'].unique())
# for sub, df in passing_tracks.groupby('reddit_sub'):

#   completed.append(sub)
#   if sub == stop_sub:
#     print(f'Stopped at {stop_sub}')
#     break

# print(f'So far completed {len(completed)} of {n_subs} ({len(completed)/n_subs:0.1%}%)')


In [14]:
completed

['2000smusic',
 '60sMusic',
 '70s',
 '70sMusic',
 '80sHipHop',
 '90shiphop',
 '90sPunk',
 '90sRock',
 'acidhouse',
 'AfricanMusic',
 'aggrotech',
 'ambientmusic',
 'atmosphericdnb',
 'baroque',
 'bassheavy',
 'bluegrass',
 'bluesrock']

In [15]:

# --- 1. Constants and Setup ---
SLEEP_TIME = 20
MIN_N_LIKE = 8
DRY = False
N_ALBUM_ENTRIES = 100

def get_tracks_from_albums(df_albums, ytm_api):
    all_track_vids = set()
    for album_id in df_albums['ytmusic_albumId'].dropna().unique():
        try:
            album_info = ytm_api.get_album(album_id)
            for track in album_info.get('tracks', []):
                if 'videoId' in track and track['videoId']:
                    all_track_vids.add(track['videoId'])
        except Exception as e:
            print(f"  - Could not fetch albumId '{album_id}'. Error: {e}")
    return all_track_vids

# --- 3. Preparation ---
yt_playlist_titles_lower = Y.playlists['title'].str.lower()

# --- 4. Main Loop ---
album_sub_grp = passing_albums.groupby('normalized_sub')
n_album_subs = len(album_sub_grp)

for i, (normalized_sub, df_group) in enumerate(album_sub_grp, start=1):
    original_subs_in_group = set(df_group['reddit_sub'].unique())
    if not original_subs_in_group.isdisjoint(completed):
        continue

    print(f"\n({i}/{n_album_subs}) Processing: '{normalized_sub}'")
    
    # A. Fetch Tracks
    new_track_vids = get_tracks_from_albums(df_group, ytm)
    new_track_vids = new_track_vids - Y.banned_vid_set
    
    if not new_track_vids:
        completed.extend(original_subs_in_group)
        continue

    # B. Apply Merge Map to Target Titles
    raw_album_title = f'xr {normalized_sub} albums'
    raw_radio_title = f'xr {normalized_sub} radio'

    # Honor merge map (check case-insensitively if needed, though map looks consistent)
    target_album_title = merge_map.get(raw_album_title, raw_album_title)
    target_radio_title = merge_map.get(raw_radio_title, raw_radio_title)

    existing_album_pl_id = None
    existing_radio_pl_id = None
    
    # Lookup merged titles in the current YT library
    if target_album_title.lower() in yt_playlist_titles_lower.values:
        idx = yt_playlist_titles_lower[yt_playlist_titles_lower == target_album_title.lower()].index[0]
        existing_album_pl_id = Y.playlists['playlistId'].loc[idx]

    if target_radio_title.lower() in yt_playlist_titles_lower.values:
        idx = yt_playlist_titles_lower[yt_playlist_titles_lower == target_radio_title.lower()].index[0]
        existing_radio_pl_id = Y.playlists['playlistId'].loc[idx]

    # C. Decision Logic
    final_pl_id = -1
    final_pl_title = ""
    vids_to_add = set()
    playlist_to_delete_id = None
    
    if existing_album_pl_id:
        pl_info = Y.playlist_get_info(existing_album_pl_id, use_cache=False)
        existing_album_tracks = {t['videoId'] for t in pl_info.get('tracks', []) if t.get('videoId')}
        total_vids = existing_album_tracks.union(new_track_vids)
        
        if len(total_vids) < N_ALBUM_ENTRIES:
            print(f"Merged target '{target_album_title}' too small. Migrating to radio.")
            final_pl_id = existing_radio_pl_id
            final_pl_title = target_radio_title
            vids_to_add = total_vids
            playlist_to_delete_id = existing_album_pl_id
        else:
            final_pl_id = existing_album_pl_id
            final_pl_title = target_album_title
            vids_to_add = new_track_vids - existing_album_tracks 
    else:
        if len(new_track_vids) < N_ALBUM_ENTRIES:
            print(f"Group too small for albums. Adding to merged radio: {target_radio_title}")
            final_pl_id = existing_radio_pl_id
            final_pl_title = target_radio_title
            vids_to_add = new_track_vids
        else:
            print(f"Creating/Updating merged album playlist: {target_album_title}")
            final_pl_id = None # New or mapped
            final_pl_title = target_album_title
            vids_to_add = new_track_vids

    # D. Execute
    if not vids_to_add:
        print("No new tracks to add.")
    elif final_pl_id:
        if not DRY: ytm.add_playlist_items(playlistId=final_pl_id, videoIds=list(vids_to_add), duplicates=False)
        print(f"Updated {len(vids_to_add)} tracks in '{final_pl_title}'")
    else:
        desc = f'Merged album tracks for {normalized_sub}'
        if not DRY: final_pl_id = ytm.create_playlist(title=final_pl_title, description=desc, privacy_status='PRIVATE', video_ids=list(vids_to_add))
        print(f"Created/Matched playlist '{final_pl_title}'")

    if playlist_to_delete_id and not DRY:
        ytm.delete_playlist(playlist_to_delete_id)

    # E. Cleanup (Handles 'Like' logic)
    if not DRY and final_pl_id and final_pl_id != -1:
        # Give the API a moment to settle if it was just created
        time.sleep(2) 
        
        try:
            # Attempt to get info with a simple retry
            pl_info = None
            for attempt in range(3):
                try:
                    pl_info = Y.playlist_get_info(final_pl_id, use_cache=False)
                    if pl_info and 'tracks' in pl_info:
                        break
                except KeyError:
                    print(f"  - Playlist data not ready yet (attempt {attempt+1}/3). Waiting...")
                    time.sleep(3)
            
            if pl_info and 'tracks' in pl_info:
                Y.clean_up_radio_playlist(
                    pl_info=pl_info,
                    verbose=True, sleep=1,
                    move_like=True, min_num_like=MIN_N_LIKE,
                    create_like_playlist=True,
                    remove_dislike=True, remove_not_like=True
                )
            else:
                print(f"  - Skipping cleanup for '{final_pl_title}': Playlist content unavailable (Ghost Playlist).")

        except Exception as e:
            print(f"  - Error during cleanup of '{final_pl_title}': {e}")
    completed.extend(original_subs_in_group)


(18/106) Processing: 'bossanova'
Group too small for albums. Adding to merged radio: xr bossanova radio
Updated 29 tracks in 'xr bossanova radio'
Playlist xr bossanova radio counters: {'moved_like': 2}

(19/106) Processing: 'brazilianmusic'
Group too small for albums. Adding to merged radio: xr brazilianmusic radio
Updated 27 tracks in 'xr brazilianmusic radio'
Playlist xr brazilianmusic radio counters: {'moved_like': 6}

(20/106) Processing: 'burial'
Group too small for albums. Adding to merged radio: xr futuregarage radio
Updated 11 tracks in 'xr futuregarage radio'
Playlist xr futuregarage radio counters: {'moved_like': 15}
  - Error during cleanup of 'xr futuregarage radio': 'NoneType' object has no attribute 'startswith'

(21/106) Processing: 'celticpunk'
Group too small for albums. Adding to merged radio: xr celticpunk radio
Updated 13 tracks in 'xr celticpunk radio'
Playlist xr celticpunk radio counters: {}

(22/106) Processing: 'chillmusic'
  - Could not fetch albumId 'MPREb_n

In [ ]:
xr surfrock radio -> rock surf radio

xr PostRock -> post rock slow core radio
xr punk-rock radio -> xr PunkRock radio